# Recover an HDS Streaming Notebook

End-to-end recovery for an HDS pipeline notebook that is stuck on a
`[DELTA_SOURCE_IGNORE_DELETE]` Delta streaming failure (typical cause:
a `TRUNCATE` on the bronze `ImagingDicom` table). The HDS streaming
consumers do not plumb `ignoreDeletes` / `skipChangeCommits`, so the
fix is to **rename each consumer's checkpoint folder under its
HealthDataManager (DMH) item** and re-trigger the owning pipeline.

This notebook codifies the manual steps used on 2026-06-02/03 to
recover:

- `healthcare1_msft_imaging_dicom_silver_metastore_transformation`
- `healthcare1_msft_imaging_dicom_fhir_conversion`

## Process

1. **Discover** — resolve notebook ID(s) from display name(s).
2. **Diagnose** — read `dbo.BusinessEvents` to confirm the v297 stream failure.
3. **Locate checkpoint** — list the DMH item under OneLake DFS.
4. **Rename checkpoint** — PUT with `x-ms-rename-source` + `?mode=legacy` quirk.
5. **Re-trigger pipeline** — Fabric pipelines `runOnDemand`.
6. **Verify** — poll job instance + re-query `BusinessEvents` and `ExecutionSummary`.

> Run cell-by-cell. The rename step is the only mutating action and is
> reversible (rename the `_bak_<date>` folder back). The downstream
> MERGE into silver is idempotent on its unique key with
> `meta_lastUpdated` precedence, so replay is safe.


## 0. Inputs

Edit these for your environment.

In [ ]:
# --- Required inputs ---
WORKSPACE_ID  = "ad5aafc2-395b-442a-9fd0-b1ba87aa067b"
DMH_ITEM_ID   = "REPLACE_WITH_HEALTHDATAMANAGER_ITEM_ID"
ADMIN_DW_FQDN = "q2idclrgtkrunbs7mfn62v3hqy-ykxvvlk3hevejh6qwg5ipkqgpm.datawarehouse.fabric.microsoft.com"
ADMIN_DW_DB   = "healthcare1_msft_admin"
PIPELINE_NAME = "prov_healthcare1_msft_imaging_clinical_foundation_with_watermark"

# Map: notebook display name -> (BusinessEvents.activityName, checkpoint path relative to DMH item root)
STUCK_CONSUMERS = {
    "healthcare1_msft_imaging_dicom_silver_metastore_transformation": {
        "activity":   "imaging_dicom_silver_metastore_transformation",
        "checkpoint": "DMHCheckpoint/medical_imaging/imagingmetastore/dicom",
    },
    "healthcare1_msft_imaging_dicom_fhir_conversion": {
        "activity":   "imaging_dicom_fhir_conversion",
        "checkpoint": "DMHCheckpoint/medical_imaging/dicom_to_fhir",
    },
}

from datetime import datetime, timezone
BACKUP_SUFFIX = "_bak_" + datetime.now(timezone.utc).strftime("%Y%m%d")
print("Backup suffix:", BACKUP_SUFFIX)


## 1. Token + helper setup

Three AAD audiences are needed:

- `https://api.fabric.microsoft.com` — Fabric REST (items, pipelines, jobs)
- `https://storage.azure.com/` — OneLake DFS (list/rename checkpoint folders)
- `https://database.windows.net/` — Fabric DW (BusinessEvents / ExecutionSummary)

The Az CLI `az storage` data plane is disabled on Fabric capacity
(`AccountIsDisabled`); REST + bearer is the only path for OneLake.


In [ ]:
import subprocess, requests, struct, time

def _az_token(resource):
    return subprocess.check_output(
        ["az", "account", "get-access-token", "--resource", resource,
         "--query", "accessToken", "-o", "tsv"]
    ).decode().strip()

def fabric_token():  return _az_token("https://api.fabric.microsoft.com")
def storage_token(): return _az_token("https://storage.azure.com/")
def dw_token():      return _az_token("https://database.windows.net/")

FABRIC_API = "https://api.fabric.microsoft.com/v1"
ONELAKE    = "https://onelake.dfs.fabric.microsoft.com"

def _hdrs(tok, extra=None):
    h = {"Authorization": f"Bearer {tok}"}
    if extra: h.update(extra)
    return h

print("Acquired tokens (lengths):")
for name, fn in [("fabric", fabric_token), ("storage", storage_token), ("dw", dw_token)]:
    try:
        print(f"  {name}: {len(fn())}")
    except Exception as e:
        print(f"  {name}: FAIL -> {e}")


## 2. Discover notebook IDs

In [ ]:
def list_notebooks(ws):
    r = requests.get(f"{FABRIC_API}/workspaces/{ws}/notebooks", headers=_hdrs(fabric_token()))
    r.raise_for_status()
    return {n["displayName"]: n["id"] for n in r.json()["value"]}

nbmap = list_notebooks(WORKSPACE_ID)
for name in STUCK_CONSUMERS:
    nb_id = nbmap.get(name)
    STUCK_CONSUMERS[name]["notebook_id"] = nb_id
    print(f"  {name} -> {nb_id}")

missing = [n for n, c in STUCK_CONSUMERS.items() if not c.get("notebook_id")]
assert not missing, f"Could not resolve notebook IDs for: {missing}"


## 3. Diagnose — query `BusinessEvents`

The Fabric job `failureReason` is generic
(`System_Cancelled_Session_Statements_Failed`). The real exception is in
`healthcare1_msft_admin.dbo.BusinessEvents`. Expected signature:

```
[STREAM_FAILED] ... [DELTA_SOURCE_IGNORE_DELETE]
Detected deleted data ... at version 297 ... at path .../Tables/ImagingDicom
```


In [ ]:
import pyodbc

def dw_connect():
    tok = dw_token()
    enc = b"".join(bytes([b, 0]) for b in tok.encode("utf-8"))
    token_struct = struct.pack(f"<I{len(enc)}s", len(enc), enc)
    conn_str = (
        f"Driver={{ODBC Driver 18 for SQL Server}};Server={ADMIN_DW_FQDN};"
        f"Database={ADMIN_DW_DB};Encrypt=yes;TrustServerCertificate=no;Connection Timeout=60"
    )
    return pyodbc.connect(conn_str, attrs_before={1256: token_struct}, timeout=60)

SELECT_ERR = (
    "SELECT activityName, severity, eventDateTime, LEFT(message, 600) AS msg "
    "FROM dbo.BusinessEvents "
    "WHERE activityName IN ({ph}) AND severity = 'error' "
    "ORDER BY eventDateTime DESC"
)

try:
    conn = dw_connect()
    cur = conn.cursor()
    acts = tuple(c["activity"] for c in STUCK_CONSUMERS.values())
    ph = ",".join("?" for _ in acts)
    cur.execute(SELECT_ERR.format(ph=ph), acts)
    rows = cur.fetchall()
    print(f"Found {len(rows)} error rows. Latest per activity:\n")
    seen = set()
    for r in rows:
        if r.activityName in seen: continue
        seen.add(r.activityName)
        print(f"--- {r.activityName} @ {r.eventDateTime} ---")
        print(r.msg); print()
    conn.close()
except Exception as e:
    print(f"BusinessEvents query failed ({e!r}). If the capacity is throttled, "
          "skip this diagnostic and proceed.")


## 4. Locate the checkpoint folders

List the DMH item under OneLake and confirm each
`STUCK_CONSUMERS[*].checkpoint` folder exists before renaming.

Notes:
- `recursive=true` is required because `directory=` filtering inside the
  query string is unreliable for the DMH layout.
- Honor `x-ms-continuation` for paged listings.


In [ ]:
def list_dfs(ws, item_id, prefix=None):
    url = f"{ONELAKE}/{ws}"
    params = {"resource": "filesystem", "recursive": "true", "directory": item_id}
    tok = storage_token()
    cont = None
    while True:
        h = _hdrs(tok, {"x-ms-version": "2023-11-03"})
        if cont: h["x-ms-continuation"] = cont
        r = requests.get(url, headers=h, params=params)
        r.raise_for_status()
        for p in r.json().get("paths", []):
            if prefix is None or p["name"].startswith(f"{item_id}/{prefix}"):
                yield p
        cont = r.headers.get("x-ms-continuation")
        if not cont: break

for name, cfg in STUCK_CONSUMERS.items():
    cp = cfg["checkpoint"]
    target = f"{DMH_ITEM_ID}/{cp}"
    hit = None
    for p in list_dfs(WORKSPACE_ID, DMH_ITEM_ID, prefix=cp):
        if p["name"] == target and p.get("isDirectory") == "true":
            hit = p; break
    cfg["checkpoint_exists"] = bool(hit)
    print(f"  {name}: {'FOUND  ' if hit else 'NOT FOUND  '}{target}")


## 5. Rename the checkpoint folders

OneLake DFS rename quirk:

- `PUT https://onelake.dfs.fabric.microsoft.com/{ws}/{NEW_PATH}?mode=legacy`
- Header `x-ms-rename-source: /{ws}/{OLD_PATH}`
- Without `?mode=legacy` you get `400 UnsupportedHeader`.

Successful rename returns `HTTP 201`.


In [ ]:
def rename_dfs(ws, old_path, new_path):
    tok = storage_token()
    url = f"{ONELAKE}/{ws}/{new_path}?mode=legacy"
    h = _hdrs(tok, {
        "x-ms-version": "2023-11-03",
        "x-ms-rename-source": f"/{ws}/{old_path}",
    })
    r = requests.put(url, headers=h)
    return r.status_code, r.text[:300]

for name, cfg in STUCK_CONSUMERS.items():
    if not cfg.get("checkpoint_exists"):
        print(f"  {name}: skip (checkpoint not found)")
        continue
    old = f"{DMH_ITEM_ID}/{cfg['checkpoint']}"
    new = f"{old}{BACKUP_SUFFIX}"
    sc, body = rename_dfs(WORKSPACE_ID, old, new)
    cfg["renamed_to"] = new if sc == 201 else None
    print(f"  {name}: HTTP {sc}  ->  {new}")
    if sc != 201:
        print(f"    body: {body}")


## 6. Re-trigger the orchestrating pipeline

In [ ]:
def find_pipeline_id(ws, name):
    r = requests.get(f"{FABRIC_API}/workspaces/{ws}/dataPipelines", headers=_hdrs(fabric_token()))
    r.raise_for_status()
    for p in r.json()["value"]:
        if p["displayName"] == name:
            return p["id"]
    return None

def run_pipeline(ws, pid):
    url = f"{FABRIC_API}/workspaces/{ws}/items/{pid}/jobs/instances?jobType=Pipeline"
    r = requests.post(url, headers=_hdrs(fabric_token(), {"Content-Type": "application/json"}), json={})
    if r.status_code not in (200, 202):
        raise RuntimeError(f"runOnDemand failed: HTTP {r.status_code} {r.text[:300]}")
    return r.headers.get("Location", "").rsplit("/", 1)[-1]

PIPELINE_ID = find_pipeline_id(WORKSPACE_ID, PIPELINE_NAME)
assert PIPELINE_ID, f"Pipeline not found: {PIPELINE_NAME}"
print(f"Pipeline: {PIPELINE_NAME} -> {PIPELINE_ID}")

JOB_INSTANCE_ID = run_pipeline(WORKSPACE_ID, PIPELINE_ID)
print(f"Triggered job instance: {JOB_INSTANCE_ID}")


## 7. Poll pipeline + each notebook

Re-run this cell to refresh status.

In [ ]:
def job_status(ws, item_id, job_id):
    r = requests.get(f"{FABRIC_API}/workspaces/{ws}/items/{item_id}/jobs/instances/{job_id}",
                     headers=_hdrs(fabric_token()))
    r.raise_for_status()
    return r.json()

def latest_run(ws, nb_id):
    r = requests.get(f"{FABRIC_API}/workspaces/{ws}/items/{nb_id}/jobs/instances?%24top=1",
                     headers=_hdrs(fabric_token()))
    r.raise_for_status()
    v = r.json()["value"]
    return v[0] if v else None

ps = job_status(WORKSPACE_ID, PIPELINE_ID, JOB_INSTANCE_ID)
print(f"Pipeline:  status={ps['status']}  start={ps['startTimeUtc']}  end={ps.get('endTimeUtc')}")
if ps.get("failureReason"):
    print(f"  failureReason: {ps['failureReason']}")

print("\nPer-notebook latest runs:")
for name, cfg in STUCK_CONSUMERS.items():
    j = latest_run(WORKSPACE_ID, cfg["notebook_id"])
    if not j:
        print(f"  {name}: no runs yet"); continue
    print(f"  {name}: status={j['status']} start={j['startTimeUtc']} end={j.get('endTimeUtc')}")
    if j.get("failureReason"):
        print(f"    failureReason: {j['failureReason']}")


## 8. Verify — no per-cell errors

Notebook snapshots are not exposed via Fabric public REST. The
authoritative source for cell-level errors in HDS notebooks is:

- `dbo.BusinessEvents` — every caught exception, even when the
  notebook overall returns `Completed`.
- `dbo.ExecutionSummary` — per-batch `executionStatus` per activity.

If both return clean for the activity names of interest, the run is
healthy.


In [ ]:
VERIFY_EVENTS = (
    "SELECT eventDateTime, activityName, severity, LEFT(message,200) AS msg "
    "FROM dbo.BusinessEvents "
    "WHERE activityName IN ({ph}) AND eventDateTime >= ? "
    "ORDER BY eventDateTime DESC"
)
VERIFY_EXEC = (
    "SELECT activityName, executionStatus, batchesProcessed, "
    "       numSourceRecords, numTargetRecords, elapsedTime, "
    "       LEFT(executionStatusDetails,200) AS details "
    "FROM dbo.ExecutionSummary "
    "WHERE activityName IN ({ph}) AND msftCreatedDatetime >= ? "
    "ORDER BY msftCreatedDatetime DESC"
)

def verify(since_iso):
    conn = dw_connect(); cur = conn.cursor()
    acts = tuple(c["activity"] for c in STUCK_CONSUMERS.values())
    ph = ",".join("?" for _ in acts)

    cur.execute(VERIFY_EVENTS.format(ph=ph), (*acts, since_iso))
    rows = cur.fetchall()
    errs = [r for r in rows if r.severity == "error"]
    print(f"BusinessEvents since {since_iso}:  total={len(rows)}  errors={len(errs)}")
    for r in errs[:5]:
        print(f"   {r.eventDateTime} [{r.activityName}] {r.msg}")

    cur.execute(VERIFY_EXEC.format(ph=ph), (*acts, since_iso))
    print(f"\nExecutionSummary since {since_iso}:")
    for r in cur.fetchall():
        flag = "OK" if r.executionStatus in ("Succeeded", "Success", "Completed") else "!!"
        print(f"  [{flag}] {r.activityName:55s} {r.executionStatus:12s} "
              f"batches={r.batchesProcessed} src={r.numSourceRecords} tgt={r.numTargetRecords}")
        if r.details: print(f"        {r.details}")
    conn.close()

# Pass the pipeline kickoff time. Format: 'YYYY-MM-DDTHH:MM:SS'
verify(ps["startTimeUtc"][:19])


## 9. Rollback (if needed)

If the rename was a mistake, swap it back. This resumes the stream
from the same parked offset (still failing).


In [ ]:
def rollback():
    for name, cfg in STUCK_CONSUMERS.items():
        new = cfg.get("renamed_to")
        if not new:
            print(f"  {name}: nothing to roll back"); continue
        orig = f"{DMH_ITEM_ID}/{cfg['checkpoint']}"
        sc, body = rename_dfs(WORKSPACE_ID, new, orig)
        print(f"  {name}: HTTP {sc}  ->  {orig}")
        if sc != 201:
            print(f"    body: {body}")

# rollback()  # uncomment to execute
print("Rollback is gated — uncomment the call above to run it.")
